# Lab 05 — Support Vector Machines and Neural Networks

**Class 05 · Topic 4.** Guide: `practice/05_svm_neural_networks.pdf`. Lecture: `slides/05_svm_neural_networks.pdf`. Demos:
`01_svm.ipynb` and `02_neural_networks.ipynb`.

Cells marked **TODO** contain `raise NotImplementedError`: replace it with your code. Every other cell is ready: it draws
the plots and prints the comparisons once your functions work.

**The rule of this lab.** The SVM, the kernel SVM, the SVR and the MLP of parts B-E are *losses written with `jax.numpy`*;
the gradient always comes from `jax.grad` (inside the given `train`). Every model is then compared with its scikit-learn
or Keras twin on the **same split and the same scaling**.

In [ ]:
import time
import warnings

import jax
import jax.numpy as jnp
import keras
import matplotlib.pyplot as plt
import numpy as np
import polars as pl
from lab05_utils import (
    compare,
    fit_standardizer,
    load_breast_cancer,
    load_digits,
    plot_decision,
    plot_digits,
    plot_history,
    plot_scores_grid,
    standardize,
    stratified_split,
    train,
)
from scipy import stats
from sklearn.datasets import make_blobs, make_moons
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics.pairwise import polynomial_kernel as sk_poly_kernel
from sklearn.metrics.pairwise import rbf_kernel as sk_rbf_kernel
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.neural_network import MLPClassifier
from sklearn.svm import SVC, SVR

warnings.filterwarnings("ignore")
plt.rcParams["figure.figsize"] = (9, 4)
pl.Config.set_tbl_rows(30)
pl.Config.set_float_precision(3)
pl.Config.set_tbl_formatting("ASCII_MARKDOWN")
_ = pl.Config.set_tbl_hide_dataframe_shape(True)

# Part A — Data visualization and preparation

Three data sets: **breast cancer** (30 measurements of cell nuclei, 569 tumours, two classes), the handwritten **digits**
(8x8 images, 1797, ten classes) and two **moons** (a 2D problem that a line cannot solve). The split is stratified and the
standardization statistics come from the **training** part only (Class 03).

In [ ]:
Xb, yb, names_b = load_breast_cancer()
Xd, yd, _ = load_digits()
print(f"breast cancer: {Xb.shape}, benign {yb.mean():.1%}; digits: {Xd.shape}, classes {np.bincount(yd)}")
rng = np.random.default_rng(0)
tr, te = stratified_split(yb, 0.25, rng)
mu, sd = fit_standardizer(Xb[tr])
Xb_tr, Xb_te = standardize(Xb[tr], mu, sd), standardize(Xb[te], mu, sd)
yb_tr, yb_te = yb[tr], yb[te]
spread = pl.DataFrame({"feature": names_b[:5], "min": Xb[:, :5].min(0), "max": Xb[:, :5].max(0), "std": Xb[:, :5].std(0)})
print(spread)

Xm, ym = make_moons(200, noise=0.25, random_state=0)
Xm_te, ym_te = make_moons(2000, noise=0.25, random_state=1)
tr_d, te_d = stratified_split(yd, 0.25, np.random.default_rng(0))
Xd_tr, Xd_te, yd_tr, yd_te = Xd[tr_d], Xd[te_d], yd[tr_d], yd[te_d]

fig, axes = plt.subplots(1, 3, figsize=(13, 3.6))
for k in (0, 1):
    axes[0].scatter(Xb_tr[yb_tr == k, 0], Xb_tr[yb_tr == k, 1], s=10, label=("malignant", "benign")[k])
axes[0].set(xlabel=f"{names_b[0]} (standardized)", ylabel=f"{names_b[1]} (standardized)", title="breast cancer, two of 30 features")
axes[0].legend()
axes[1].scatter(*Xm.T, c=ym, cmap="coolwarm", s=12)
axes[1].set_title("moons")
corr = np.corrcoef(Xb_tr.T)
im = axes[2].imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)
axes[2].set_title("breast cancer: correlation of the 30 features")
plt.colorbar(im, ax=axes[2])
plt.tight_layout()
plt.show()
plot_digits(Xd, yd)

# Part B — The linear SVM as a loss

With labels $y_i \in \{-1, +1\}$ and $f(x) = w^\top x + b$, the soft-margin SVM minimizes the **hinge loss plus an $L_2$
penalty**:

$$\mathcal{L}(w, b) = \frac1n\sum_i \max\big(0,\ 1 - y_i f(x_i)\big) + \frac{\lambda}{2}\lVert w\rVert^2,
\qquad \lambda = \frac{1}{C\,n}.$$

## B1. The loss and its support vectors

Implement `hinge_loss(params, X, y, lam)` with `params = {"w": ..., "b": ...}` (no gradient formula: `jax.grad` is called
by `train`), `margins(params, X, y)` $= y_i f(x_i)$, and `support_mask(params, X, y, tol)`: the points with margin $\le 1 +$ `tol`.

In [ ]:
def hinge_loss(params, X, y, lam):
    raise NotImplementedError


def margins(params, X, y):
    raise NotImplementedError


def support_mask(params, X, y, tol=1e-3):
    raise NotImplementedError

The driver trains the SVM for three values of $C$ (`lam = 1 / (C n)`) on the breast-cancer data and compares the
weights, the predictions and the number of support vectors with `SVC(kernel="linear")`, a quadratic-programming solver.

In [ ]:
ypm_tr = 2.0 * yb_tr - 1
ypm_te = 2.0 * yb_te - 1
rows = []
for C in (0.01, 0.1, 1, 10):
    lam = 1 / (C * len(yb_tr))
    p0 = {"w": jnp.zeros(Xb_tr.shape[1]), "b": jnp.zeros(())}
    p, hist = train(hinge_loss, p0, Xb_tr, ypm_tr, lam, lr=0.05, steps=3000)
    ref = SVC(kernel="linear", C=C, tol=1e-8).fit(Xb_tr, yb_tr)
    f_te = np.asarray(Xb_te @ p["w"] + p["b"])
    rows.append((C, float(np.abs(np.asarray(p["w"]) - ref.coef_[0]).max()), float(np.mean((f_te > 0) == ref.predict(Xb_te))),
                 int(support_mask(p, Xb_tr, ypm_tr).sum()), len(ref.support_), float(np.mean((f_te > 0) == yb_te))))
print(pl.DataFrame(rows, schema=["C", "max |w - w_svc|", "same predictions", "SV (ours)", "SV (SVC)", "test accuracy"], orient="row"))

**The picture in two dimensions.** Three values of $C$ on two Gaussian blobs: the dashed lines are $f = \pm 1$; the rings
are the support vectors of your loss.

In [ ]:
X2, y2 = make_blobs(120, centers=[(-1.0, -0.8), (1.0, 0.8)], cluster_std=1.2, random_state=1)
y2pm = 2.0 * y2 - 1
fig, axes = plt.subplots(1, 3, figsize=(12, 3.6))
for ax, C in zip(axes, (0.01, 1, 100)):
    p, _ = train(hinge_loss, {"w": jnp.zeros(2), "b": jnp.zeros(())}, X2, y2pm, 1 / (C * len(y2)), lr=0.05, steps=3000)
    sv = support_mask(p, X2, y2pm)
    w_, b_ = np.asarray(p["w"]), float(p["b"])
    plot_decision(lambda P, w_=w_, b_=b_: P @ w_ + b_, X2, y2, f"C = {C:g}: {sv.sum()} support vectors, width {2 / np.linalg.norm(w_):.2f}",
                  ax=ax, sv=X2[sv], margin=True)
plt.tight_layout()
plt.show()

# Part C — Kernels

A kernel $k(a, b)$ is a similarity. With the Gram matrix $K_{ij} = k(x_i, x_j)$, the SVM in its **representer form** is

$$f(x) = \sum_j \beta_j\, k(x_j, x) + b, \qquad
\mathcal{L}(\beta, b) = \frac1n\sum_i \max\big(0,\ 1 - y_i f(x_i)\big) + \frac{\lambda}{2}\,\beta^\top K\beta .$$

## C1. The kernels and the loss

Implement, with broadcasting and no Python loop over the points: `rbf_kernel(A, B, gamma)` $= \exp(-\gamma\lVert a - b\rVert^2)$
for all pairs (shape `(len(A), len(B))`), `poly_kernel(A, B, degree, coef0)` $= (a^\top b + c_0)^d$, and
`kernel_svm_loss(params, K, y, lam)` with `params = {"beta": ..., "b": ...}` and `K` the Gram matrix of the training set.
*Hint:* $\lVert a - b\rVert^2 = \lVert a\rVert^2 + \lVert b\rVert^2 - 2\,a^\top b$.

In [ ]:
def rbf_kernel(A, B, gamma):
    raise NotImplementedError


def poly_kernel(A, B, degree, coef0):
    raise NotImplementedError


def kernel_svm_loss(params, K, y, lam):
    raise NotImplementedError

In [ ]:
A_, B_ = jnp.asarray(Xm[:50]), jnp.asarray(Xm[50:90])
compare("rbf kernel (gamma 0.7)", rbf_kernel(A_, B_, 0.7), sk_rbf_kernel(Xm[:50], Xm[50:90], gamma=0.7))
compare("polynomial kernel (d 3, c0 1)", poly_kernel(A_, B_, 3, 1.0), sk_poly_kernel(Xm[:50], Xm[50:90], degree=3, gamma=1.0, coef0=1.0))

ympm = 2.0 * ym - 1
C, gamma = 10.0, 1.0
K_tr = rbf_kernel(jnp.asarray(Xm), jnp.asarray(Xm), gamma)
K_te = rbf_kernel(jnp.asarray(Xm_te), jnp.asarray(Xm), gamma)
p, hist = train(kernel_svm_loss, {"beta": jnp.zeros(len(ym)), "b": jnp.zeros(())}, K_tr, jnp.asarray(ympm), 1 / (C * len(ym)),
                lr=0.02, steps=4000)
f_ours = np.asarray(K_te @ p["beta"] + p["b"])
ref = SVC(kernel="rbf", C=C, gamma=gamma).fit(Xm, ym)
print(f"test accuracy: ours {np.mean((f_ours > 0) == ym_te):.3f}   SVC {ref.score(Xm_te, ym_te):.3f}   "
      f"same prediction on {np.mean((f_ours > 0) == ref.predict(Xm_te)):.1%} of the test points")
print(f"correlation of the decision functions: {np.corrcoef(f_ours, ref.decision_function(Xm_te))[0, 1]:.4f}")

**Gamma.** The same loss for four values of $\gamma$ (training accuracy, test accuracy, number of non-negligible $\beta_j$):

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(15, 3.4))
rows = []
for ax, g in zip(axes, (0.1, 1, 10, 100)):
    Kt = rbf_kernel(jnp.asarray(Xm), jnp.asarray(Xm), g)
    pg, _ = train(kernel_svm_loss, {"beta": jnp.zeros(len(ym)), "b": jnp.zeros(())}, Kt, jnp.asarray(ympm), 1 / (C * len(ym)), lr=0.02, steps=4000)
    def score(P, pg=pg, g=g):
        return np.asarray(rbf_kernel(jnp.asarray(P), jnp.asarray(Xm), g) @ pg["beta"] + pg["b"])

    acc_tr = np.mean((score(Xm) > 0) == ym)
    acc_te = np.mean((score(Xm_te) > 0) == ym_te)
    nz = int(np.sum(np.abs(np.asarray(pg["beta"])) > 0.05 * np.abs(np.asarray(pg["beta"])).max()))
    rows.append((g, float(acc_tr), float(acc_te), nz))
    plot_decision(score, Xm, ym, f"gamma = {g:g}", ax=ax)
plt.tight_layout()
plt.show()
print(pl.DataFrame(rows, schema=["gamma", "train acc", "test acc", "large |beta_j|"], orient="row"))

**Choosing $C$ and $\gamma$.** The log-spaced grid with 5-fold cross-validation (scikit-learn's `GridSearchCV`, given) on the
breast-cancer data: with raw features and with the standardized ones.

In [ ]:
Cs, gs = 10.0 ** np.arange(-2, 4), 10.0 ** np.arange(-6, 1)
fig_scores = {}
for label, (A, y_) in {"raw features": (Xb[tr], yb_tr), "standardized": (Xb_tr, yb_tr)}.items():
    t0 = time.time()
    grid = GridSearchCV(SVC(), {"C": Cs, "gamma": gs}, cv=StratifiedKFold(5, shuffle=True, random_state=0), n_jobs=-1).fit(A, y_)
    fig_scores[label] = grid.cv_results_["mean_test_score"].reshape(len(Cs), len(gs))
    print(f"{label:<14}: best {grid.best_params_}, CV accuracy {grid.best_score_:.3f}   ({time.time() - t0:.1f} s)")
for label, sc in fig_scores.items():
    plot_scores_grid(Cs, gs, sc, "gamma (log)", "C (log)", f"CV accuracy, {label}")

# Part D — Support vector regression

For a numeric target the hinge becomes the **$\varepsilon$-insensitive loss**: errors smaller than $\varepsilon$ cost
nothing, larger ones cost linearly.

## D1. The loss

Implement `eps_insensitive(r, eps)` $= \max(0, |r| - \varepsilon)$ and
`svr_loss(params, X, y, eps, lam)` $=$ mean of `eps_insensitive(y - (X w + b), eps)` $+ \tfrac{\lambda}{2}\lVert w\rVert^2$
with `params = {"w": ..., "b": ...}`.

In [ ]:
def eps_insensitive(r, eps):
    raise NotImplementedError


def svr_loss(params, X, y, eps, lam):
    raise NotImplementedError

A noisy line ($y = 2x + 1$, noise 1) with **three gross outliers on the same side**. Least squares is pulled by them;
the $\varepsilon$-insensitive loss (like the absolute error) is not.

In [ ]:
rng = np.random.default_rng(0)
x1 = np.linspace(0, 10, 40)
y1 = 2 * x1 + 1 + rng.normal(0, 1, 40)
y1[[5, 20, 33]] += [25.0, 30.0, 28.0]
X1 = x1[:, None]
p_svr, _ = train(svr_loss, {"w": jnp.zeros(1), "b": jnp.zeros(())}, X1, y1, 0.5, 1 / (10 * len(y1)), lr=0.1, steps=4000)
ols = LinearRegression().fit(X1, y1)
sk = SVR(kernel="linear", C=10, epsilon=0.5).fit(X1, y1)
print(pl.DataFrame({"model": ["least squares", "ours (jax.grad)", "SVR (scikit-learn)"],
                    "slope": [float(ols.coef_[0]), float(p_svr["w"][0]), float(sk.coef_[0, 0])],
                    "intercept": [float(ols.intercept_), float(p_svr["b"]), float(sk.intercept_[0])]}))
plt.scatter(x1, y1, s=12)
xx = np.linspace(0, 10, 2)
plt.plot(xx, ols.coef_[0] * xx + ols.intercept_, label="least squares")
plt.plot(xx, float(p_svr["w"][0]) * xx + float(p_svr["b"]), label="epsilon-insensitive")
plt.legend()
plt.show()

# Part E — A multilayer perceptron in JAX (a practice exercise)

Keras (Part F) is the tool you will use; here you write the smallest network yourself, to see that there is nothing more
to it than *parameters, a forward pass and a loss*. The parameters are a **list of dictionaries**
`[{"W": ..., "b": ...}, ...]`, one per layer: a pytree, so `jax.grad` returns a gradient with the same structure.

## E1. Initialization, forward pass, loss

Implement

* `init_params(key, sizes)`: for `sizes = [64, 32, 10]` two layers with $W \sim \mathcal{N}(0, 2/n_\text{in})$ (He) and
  $b = 0$ (use `jax.random.split` and `jax.random.normal`);
* `forward(params, X)`: ReLU after every layer except the last; returns the **logits**;
* `cross_entropy(params, X, y)`: the mean of $-\log\text{softmax}(\text{logits})_{y}$, computed stably as
  `logsumexp(logits) - logits[y]` (`y` holds the class index).

In [ ]:
def init_params(key, sizes):
    raise NotImplementedError


def forward(params, X):
    raise NotImplementedError


def cross_entropy(params, X, y):
    raise NotImplementedError

**XOR and the digits.** A network without hidden layer cannot solve XOR; one hidden layer can. Then the digits: 64 pixels,
a hidden layer of 32 units, 10 classes, full-batch Adam through `train`.

In [ ]:
rng = np.random.default_rng(0)
Xx = rng.uniform(0.15, 1.0, (400, 2)) * rng.choice([-1.0, 1.0], (400, 2))
yx = (Xx[:, 0] * Xx[:, 1] < 0).astype(int)
fig, axes = plt.subplots(1, 2, figsize=(9, 3.6))
for ax, sizes in zip(axes, ([2, 2], [2, 8, 2])):
    p, _ = train(cross_entropy, init_params(jax.random.PRNGKey(0), sizes), jnp.asarray(Xx), jnp.asarray(yx), lr=0.05, steps=1500)
    acc = np.mean(np.asarray(forward(p, jnp.asarray(Xx)).argmax(1)) == yx)
    plot_decision(lambda P, p=p: np.asarray(forward(p, jnp.asarray(P))[:, 1] - forward(p, jnp.asarray(P))[:, 0]), Xx, yx,
                  f"layers {sizes}: accuracy {acc:.2f}", ax=ax)
    print(f"XOR, layers {sizes}: {sum(int(np.prod(q['W'].shape)) + int(np.prod(q['b'].shape)) for q in p)} parameters, accuracy {acc:.3f}")
plt.tight_layout()
plt.show()

params = init_params(jax.random.PRNGKey(1), [64, 32, 10])
print("initial loss:", float(cross_entropy(params, jnp.asarray(Xd_tr), jnp.asarray(yd_tr))), "(random guess: log 10 = 2.303)")
t0 = time.time()
params, hist_jax = train(cross_entropy, params, jnp.asarray(Xd_tr), jnp.asarray(yd_tr), lr=0.01, steps=1000)
acc_jax = float(np.mean(np.asarray(forward(params, jnp.asarray(Xd_te)).argmax(1)) == yd_te))
time_jax = time.time() - t0
print(f"plain JAX MLP [64, 32, 10]: {time_jax:.1f} s, train loss {hist_jax[-1]:.4f}, test accuracy {acc_jax:.3f}")
plot_history({"training loss": hist_jax})

## E2. Backpropagation by hand

For the network with **one** ReLU hidden layer, `params = [{"W": W1, "b": b1}, {"W": W2, "b": b2}]`, implement
`backward_by_hand(params, X, y)` returning a list with the same structure as `params` (the gradient of `cross_entropy`).
Follow the chain rule from the loss to the input; the forward values you need are $z_1 = XW_1 + b_1$, $h = \text{ReLU}(z_1)$,
the logits $z_2 = hW_2 + b_2$ and $p = \text{softmax}(z_2)$.

*Hints:* $\partial\mathcal{L}/\partial z_2 = (p - \text{onehot}(y))/n$; for $z = hW + b$: $\partial/\partial W = h^\top \delta$,
$\partial/\partial b = \sum_i \delta_i$, $\partial/\partial h = \delta W^\top$; the ReLU passes the gradient where $z_1 > 0$.

In [ ]:
def backward_by_hand(params, X, y):
    raise NotImplementedError

In [ ]:
small = init_params(jax.random.PRNGKey(2), [64, 16, 10])
Xj, yj = jnp.asarray(Xd_tr[:200]), jnp.asarray(yd_tr[:200])
g_hand = backward_by_hand(small, Xj, yj)
g_auto = jax.grad(cross_entropy)(small, Xj, yj)
for i, (gh, ga) in enumerate(zip(g_hand, g_auto)):
    compare(f"layer {i + 1}: dW", gh["W"], ga["W"])
    compare(f"layer {i + 1}: db", gh["b"], ga["b"])

## E3. Three implementations of the same network

The network of E1 (`[64, 32, 10]`: 64 pixels, 32 ReLU units, 10 classes) written in the three tools you meet in this course:
your JAX code, scikit-learn's `MLPClassifier` and a Keras `Sequential` (JAX backend). Same split, same scaling, same
optimizer (Adam, learning rate 0.01), full batch and 1000 steps. There is no code to write: the driver prints the number of
parameters, the training loss, the test accuracy and how often two models give the same answer.


In [ ]:
if "time_jax" not in globals():
    raise NotImplementedError("finish E1 first: this cell compares your trained network")

Xtr_j, ytr_j = jnp.asarray(Xd_tr), jnp.asarray(yd_tr)
n_tr = len(yd_tr)

t0 = time.time()
sk = MLPClassifier((32,), solver="adam", learning_rate_init=0.01, batch_size=n_tr, max_iter=1000, tol=0.0,
                   n_iter_no_change=1000, alpha=0.0, random_state=0).fit(Xd_tr, yd_tr)
t_sk = time.time() - t0

keras.utils.set_random_seed(0)
kmodel = keras.Sequential([keras.Input((64,)), keras.layers.Dense(32, activation="relu"),
                           keras.layers.Dense(10, activation="softmax")])
kmodel.compile(keras.optimizers.Adam(0.01), "sparse_categorical_crossentropy")
t0 = time.time()
kmodel.fit(Xd_tr, yd_tr, epochs=1000, batch_size=n_tr, verbose=0)
t_keras = time.time() - t0

p_sk = sk.predict_proba(Xd_tr)
loss = {
    "plain JAX (Part E)": float(cross_entropy(params, Xtr_j, ytr_j)),
    "scikit-learn MLPClassifier": float(-np.mean(np.log(p_sk[np.arange(n_tr), yd_tr]))),
    "Keras Sequential (JAX backend)": float(kmodel.evaluate(Xd_tr, yd_tr, verbose=0)),
}
pred = {
    "plain JAX (Part E)": np.asarray(forward(params, jnp.asarray(Xd_te)).argmax(1)),
    "scikit-learn MLPClassifier": sk.predict(Xd_te),
    "Keras Sequential (JAX backend)": kmodel.predict(Xd_te, verbose=0).argmax(1),
}
n_par = {
    "plain JAX (Part E)": sum(int(np.prod(q["W"].shape)) + int(np.prod(q["b"].shape)) for q in params),
    "scikit-learn MLPClassifier": sum(c.size for c in sk.coefs_) + sum(i.size for i in sk.intercepts_),
    "Keras Sequential (JAX backend)": kmodel.count_params(),
}
secs = {"plain JAX (Part E)": float(time_jax), "scikit-learn MLPClassifier": t_sk, "Keras Sequential (JAX backend)": t_keras}
rows = [(k, n_par[k], loss[k], float(np.mean(pred[k] == yd_te)), secs[k]) for k in pred]
print(pl.DataFrame(rows, schema=["implementation", "parameters", "train loss", "test accuracy", "time (s)"], orient="row"))
names = list(pred)
for i in range(3):
    for j in range(i + 1, 3):
        print(f"same prediction, {names[i]} vs {names[j]}: {np.mean(pred[names[i]] == pred[names[j]]):.1%}")

# Part F — Networks with Keras

Keras 3 on the JAX backend is the tool for real work: `Sequential`, `compile` (optimizer and loss), `fit`.

## F1. Build the network

Implement `build_keras_mlp(sizes, n_in, n_out, l2=0.0, dropout=0.0, seed=0)`: a `keras.Sequential` with
`keras.Input((n_in,))`, one `Dense(size, activation="relu", kernel_regularizer=L2(l2) if l2 else None)` per hidden size
(followed by `Dropout(dropout)` if `dropout > 0`), and a final `Dense(n_out, activation="softmax")`.
Call `keras.utils.set_random_seed(seed)` first, so that the weights are reproducible.

In [ ]:
def build_keras_mlp(sizes, n_in, n_out, l2=0.0, dropout=0.0, seed=0):
    raise NotImplementedError

The driver trains five networks on the digits (training part split again into 80% fit and 20% validation) and compares
them with the plain JAX network of Part E and with scikit-learn's `MLPClassifier`.

In [ ]:
n_fit = int(0.8 * len(yd_tr))
Xf, yf, Xv, yv = Xd_tr[:n_fit], yd_tr[:n_fit], Xd_tr[n_fit:], yd_tr[n_fit:]
configs = {
    "no regularization": dict(),
    "weight decay 1e-3": dict(l2=1e-3),
    "dropout 0.3": dict(dropout=0.3),
    "early stopping": dict(early=True),
    "dropout + early stopping": dict(dropout=0.3, early=True),
}
histories, rows = {}, []
for name, cfg in configs.items():
    early = cfg.pop("early", False)
    net = build_keras_mlp([64, 64], 64, 10, seed=0, **cfg)
    net.compile(keras.optimizers.Adam(1e-3), "sparse_categorical_crossentropy", metrics=["accuracy"])
    cb = [keras.callbacks.EarlyStopping(monitor="val_loss", patience=8, restore_best_weights=True)] if early else []
    t0 = time.time()
    h = net.fit(Xf, yf, epochs=100, batch_size=32, validation_data=(Xv, yv), verbose=0, callbacks=cb)
    histories[name] = np.array(h.history["val_loss"])
    rows.append((name, len(h.history["loss"]), float(h.history["accuracy"][-1]), float(net.evaluate(Xd_te, yd_te, verbose=0)[1]), time.time() - t0))
skl = MLPClassifier((64, 64), max_iter=300, random_state=0).fit(Xf, yf)
rows.append(("scikit-learn MLPClassifier", skl.n_iter_, float(skl.score(Xf, yf)), float(skl.score(Xd_te, yd_te)), np.nan))
rows.append(("plain JAX (Part E, 32 units)", 1000, float(np.mean(np.asarray(forward(params, jnp.asarray(Xd_tr)).argmax(1)) == yd_tr)), acc_jax, np.nan))
print(pl.DataFrame(rows, schema=["network", "epochs", "train acc", "test acc", "time (s)"], orient="row"))
plot_history(histories, ylabel="validation loss", logy=False)

**Seeds.** The same network and data, ten seeds: the test accuracy is a random variable (Class 01).

In [ ]:
accs = []
for seed in range(10):
    net = build_keras_mlp([64, 64], 64, 10, seed=seed)
    net.compile(keras.optimizers.Adam(1e-3), "sparse_categorical_crossentropy")
    net.fit(Xf, yf, epochs=30, batch_size=64, verbose=0)
    accs.append(float(np.mean(net.predict(Xd_te, verbose=0).argmax(1) == yd_te)))
print(f"test accuracy over 10 seeds: mean {np.mean(accs):.4f}, sd {np.std(accs, ddof=1):.4f}, min {np.min(accs):.3f}, max {np.max(accs):.3f}")

# Part G — Challenge: which learner? (home)

Choose a data set of `datasets/` (`german_credit`, `wine_quality`, `heart_disease`, `spambase`; see
`datasets/README.md`), prepare it with polars (missing values, one-hot encoding, a binary target), and compare **logistic
regression**, the **RBF SVM** and a **Keras MLP** on 10 stratified 75/25 splits with the same standardization, *fitted on the
training part*. Judge the difference with the **corrected resampled $t$-test** (Class 01):
$t = \bar d \big/ \sqrt{(1/J + n_\text{test}/n_\text{train})\,s_d^2}$ with $J-1$ degrees of freedom.

## G1. The test

Implement `corrected_ttest(a, b, ratio)` returning `(t, p)` for the paired scores `a`, `b` of $J$ repetitions
(`ratio = n_test / n_train`; two-sided $p$ from `scipy.stats.t`).

In [ ]:
def corrected_ttest(a, b, ratio):
    raise NotImplementedError

In [ ]:
Xg, yg_raw, _ = __import__("lab05_utils").load_table("wine_quality", "quality")
yg = (yg_raw >= 6).astype(int)
scores = {"logistic": [], "SVM rbf": [], "MLP": []}
for rep in range(10):
    a_idx, b_idx = stratified_split(yg, 0.25, np.random.default_rng(rep))
    m_, s_ = fit_standardizer(Xg[a_idx])
    A, B = standardize(Xg[a_idx], m_, s_), standardize(Xg[b_idx], m_, s_)
    scores["logistic"].append(LogisticRegression(max_iter=2000).fit(A, yg[a_idx]).score(B, yg[b_idx]))
    scores["SVM rbf"].append(SVC().fit(A, yg[a_idx]).score(B, yg[b_idx]))
    net = build_keras_mlp([64, 64], A.shape[1], 2, dropout=0.2, seed=rep)
    net.compile(keras.optimizers.Adam(1e-3), "sparse_categorical_crossentropy")
    net.fit(A, yg[a_idx], epochs=30, batch_size=64, verbose=0)
    scores["MLP"].append(float(np.mean(net.predict(B, verbose=0).argmax(1) == yg[b_idx])))
ratio = len(b_idx) / len(a_idx)
rows = [(k, float(np.mean(v)), float(np.std(v, ddof=1))) for k, v in scores.items()]
print(pl.DataFrame(rows, schema=["model (wine quality >= 6)", "mean accuracy", "sd"], orient="row"))
for a, b in (("SVM rbf", "MLP"), ("SVM rbf", "logistic"), ("MLP", "logistic")):
    t, p = corrected_ttest(scores[a], scores[b], ratio)
    print(f"{a} - {b}: mean difference {np.mean(scores[a]) - np.mean(scores[b]):+.4f}, t = {t:.2f}, p = {p:.3f}")